In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.embedding.natural_text


>DOM embedding
output-file: core.dom.embedding.natural_text
title: core.dom.embedding.natural_text

This notebook provides utility functions for DOM embedding
---

In [ ]:
# | default_exp core.dom.embedding.natural_text

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
from fastcore.test import *


In [ ]:
# | export
import json
import os
import re
from collections.abc import Callable
from pathlib import Path
from typing import Optional, Any
import hashlib

import numpy as np
from ollama import AsyncClient

from chromadb.api import ClientAPI
from chromadb.api.models.Collection import Collection

In [ ]:
# | hide
import asyncio
import threading

In [ ]:
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide

def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, object] = {}
    error: dict[str, BaseException] = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')

class FakeEmbedResponse:
    def __init__(self, embeddings=None, model="fake-model"):
        self.embeddings = embeddings
        self.model = model


class FakeAsyncClient:
    def __init__(self, responses):
        self.responses = list(responses)
        self.calls = []

    async def embed(self, model: str, input: str):
        self.calls.append({"model": model, "input": input})
        if not self.responses:
            raise AssertionError("No fake responses remaining")
        response = self.responses.pop(0)
        return response() if callable(response) else response


class FakeCollection:
    def __init__(self):
        self.rows = {}
        self.operations = []

    @property
    def records(self):
        return list(self.rows.values())

    def upsert(self, ids, metadatas, embeddings, documents):
        self.operations.append(('upsert', list(ids)))
        for i, identifier in enumerate(ids):
            self.rows[identifier] = dict(ids=[identifier], metadatas=[metadatas[i]],
                embeddings=embeddings[i:i+1], documents=[documents[i]])

    def get(self, where=None, include=None):
        self.operations.append(('get', where))
        return {'ids': [identifier for identifier, row in self.rows.items()
            if not where or all(row['metadatas'][0].get(k) == v for k, v in where.items())]}

    def delete(self, ids):
        self.operations.append(('delete', list(ids)))
        for identifier in ids:
            self.rows.pop(identifier, None)


In [ ]:
# | export
def normalize_embeddings(raw_embeddings: list[list[float]] | list[float] | None) -> list[list[float]] | None:
    """Normalize Ollama embedding payloads to Chroma-compatible list[list[float]]."""
    if raw_embeddings is None:
        return None
    if isinstance(raw_embeddings, dict):
        raw_embeddings = raw_embeddings.get('embeddings') or raw_embeddings.get('embedding')
    # if hasattr(raw_embeddings, 'tolist'):
    #     raw_embeddings = raw_embeddings.tolist()
    if isinstance(raw_embeddings, list):
        if not raw_embeddings:
            return None
        if isinstance(raw_embeddings[0], (int, float)):
            return [[float(x) for x in raw_embeddings]]
        if isinstance(raw_embeddings[0], list):
            normalized = []
            for row in raw_embeddings:
                if isinstance(row, list) and row:
                    normalized.append([float(x) for x in row])
            return normalized or None
    return None


In [ ]:
# | hide
def test_normalize_embeddings_handles_supported_shapes():
    test_eq(normalize_embeddings(None), None)
    test_eq(normalize_embeddings([]), None)
    test_eq(normalize_embeddings([1, 2.5, 3]), [[1.0, 2.5, 3.0]])
    test_eq(normalize_embeddings([[1, 2], [3.5, 4]]), [[1.0, 2.0], [3.5, 4.0]])
    test_eq(normalize_embeddings({"embedding": [1, 2]}), [[1.0, 2.0]])
    test_eq(normalize_embeddings({"embeddings": [[1, 2], [], [3, 4]]}), [[1.0, 2.0], [3.0, 4.0]])
    test_eq(normalize_embeddings("invalid"), None)


test_normalize_embeddings_handles_supported_shapes()

In [ ]:
# | export
async def get_embeddings_with_retry(text: str, item_label: str, client:AsyncClient, model:str="bge-m3") -> tuple[list[float] | None, Any]:
    """Fetch embeddings and retry once if payload is empty/invalid.

    Args:
        text: The input text to embed.
        item_label: A label for the item being embedded, used for logging.
        client: The AsyncClient instance to use for embedding.
        model: The embedding model to use.

    Returns:
        A tuple of (embeddings, response). If embeddings are empty, returns (None, response).
    """
    response = await client.embed(model=model, input=text)
    raw_embeddings: list[list[float]] | list[float] | None = None

    if isinstance(response, dict):
        candidate = response.get("embeddings") or response.get("embedding")
        if isinstance(candidate, list):
            raw_embeddings = candidate
    elif isinstance(response, list):
        raw_embeddings = response
    else:
        candidate = getattr(response, "embeddings", None)
        if isinstance(candidate, list):
            raw_embeddings = candidate

    embeddings = normalize_embeddings(raw_embeddings)
    if embeddings:
        return embeddings[0], response

    # Small retry for transient empty payloads
    response_retry = await client.embed(model=model, input=text)
    raw_embeddings_retry: list[list[float]] | list[float] | None = None

    if isinstance(response_retry, dict):
        candidate_retry = response_retry.get("embeddings") or response_retry.get("embedding")
        if isinstance(candidate_retry, list):
            raw_embeddings_retry = candidate_retry
    elif isinstance(response_retry, list):
        raw_embeddings_retry = response_retry
    else:
        candidate_retry = getattr(response_retry, "embeddings", None)
        if isinstance(candidate_retry, list):
            raw_embeddings_retry = candidate_retry

    normalized_retry = normalize_embeddings(raw_embeddings_retry)
    embeddings_retry = normalized_retry[0] if normalized_retry else None
    if embeddings_retry:
        print(f"Embedding retry succeeded for {item_label}")
        return embeddings_retry, response_retry

    print(f"Embedding retry failed for {item_label}; payload remains empty/invalid")
    return None, response_retry

In [ ]:
# | hide
def test_get_embeddings_with_retry_returns_first_valid_payload():
    client = FakeAsyncClient([
        {"embeddings": [[0.1, 0.2, 0.3]]},
    ])

    embeddings, response = run_async(
        get_embeddings_with_retry("hello", "item-1", client=client, model="m1")  # type: ignore
    )

    test_eq(embeddings, [0.1, 0.2, 0.3])
    test_eq(response["embeddings"], [[0.1, 0.2, 0.3]])
    test_eq(len(client.calls), 1)


def test_get_embeddings_with_retry_retries_once_then_succeeds():
    client = FakeAsyncClient([
        {"embedding": []},
        FakeEmbedResponse([[1, 2, 3]], model="retry-model"),
    ])

    embeddings, response = run_async(
        get_embeddings_with_retry("hello", "item-2", client=client, model="m2")  # type: ignore
    )

    test_eq(embeddings, [1.0, 2.0, 3.0])
    test_eq(response.model, "retry-model")
    test_eq(len(client.calls), 2)


def test_get_embeddings_with_retry_returns_none_after_failed_retry():
    client = FakeAsyncClient([
        {"embeddings": []},
        FakeEmbedResponse(None),
    ])

    embeddings, response = run_async(
        get_embeddings_with_retry("hello", "item-3", client=client, model="m3")  # type: ignore
    )

    test_eq(embeddings, None)
    test_eq(response.embeddings, None)
    test_eq(len(client.calls), 2)


test_get_embeddings_with_retry_returns_first_valid_payload()
test_get_embeddings_with_retry_retries_once_then_succeeds()
test_get_embeddings_with_retry_returns_none_after_failed_retry()

In [ ]:
# | export
def _canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ';'))


def _semantic_payload(value):
    """Strip only recognizable pipeline annotations; retain business i/m fields."""
    if isinstance(value, dict):
        metadata = value.get('m')
        owned = (isinstance(value.get('i'), str)
                 and re.fullmatch(r'[0-9a-f]{64}', value['i']) is not None
                 and isinstance(metadata, dict)
                 and {'embed_model', 'node_type', 'node_path'} <= metadata.keys()
                 and ('document_id' in metadata or 'doc_id' in metadata)
                 and (isinstance(value.get('t'), str) or 'summary' in value or 'blocks' in value
                      or (metadata.get('node_type') == 'Document'
                          and metadata.get('document_id', metadata.get('doc_id')) == value['i']
                          and metadata.get('node_path') in ('$', value['i']))))
        return {k: _semantic_payload(v) for k, v in value.items()
                if not (owned and k in ('i', 'm'))}
    if isinstance(value, list):
        return [_semantic_payload(v) for v in value]
    return value


def _child_path(parent, key_or_index):
    return parent + '/' + str(key_or_index).replace('~', '~0').replace('/', '~1')


def _response_model(response, requested_model):
    value = response.get('model') if isinstance(response, dict) else getattr(response, 'model', None)
    return value if isinstance(value, str) and value else requested_model


def _validate_embed_types(embed_types):
    if isinstance(embed_types, str) or not all(isinstance(t, str) for t in embed_types):
        raise ValueError('embed_types must contain strings')


def _transform_embedding_tree(value, action, path='$'):
    """Apply action once per dict/list (including list containers), before hashing.

    Scalar fields are preserved and are not callback targets, matching the public
    embedding callback contract. Replacements must retain their container category.
    """
    if isinstance(value, (dict, list)):
        replacement = action(value) if action is not None else value
        if not isinstance(replacement, type(value)):
            raise ValueError(f'action must preserve {type(value).__name__} at {path}')
        if isinstance(replacement, dict):
            return {k: _transform_embedding_tree(v, action, _child_path(path, k))
                    for k, v in replacement.items()}
        return [_transform_embedding_tree(v, action, _child_path(path, i))
                for i, v in enumerate(replacement)]
    return value


def _node_id(node, doc_id, structural_path):
    framed = {'document_id': doc_id, 'node_path': structural_path,
              'semantic_node': _semantic_payload(node)}
    return hashlib.sha256(_canonical_json(framed).encode('utf-8')).hexdigest()


async def _persist_embedding(node, node_id, doc_id, structural_path, file_path,
                             collection, llm_client, model, written, root=False):
    text = node.get('s') or (node.get('summary', '') if root else '')
    if not isinstance(text, str):
        raise ValueError(f'Embedding summary must be a string at {structural_path}')
    if not text.strip():
        return node
    try:
        vector, response = await get_embeddings_with_retry(
            text, f'{file_path} node {node_id} at {structural_path}', llm_client, model)
        if not vector or not np.isfinite(vector).all():
            raise ValueError('empty or non-finite embedding')
        metadata = {'embed_model': _response_model(response, model),
                    'node_type': node.get('t') or ('Document' if root else ''),
                    'node_path': structural_path}
        if doc_id:
            metadata['document_id'] = doc_id
        if file_path:
            metadata['file_path'] = file_path
        collection.upsert(ids=[node_id], metadatas=[metadata],
                          embeddings=np.asarray([vector], dtype=np.float32), documents=[text])
    except Exception as exc:
        raise RuntimeError(f'Embedding failed for document {doc_id}, source {file_path}, '
                           f'node {node_id} at {structural_path}: {exc}') from exc
    written.add(node_id)
    node['i'], node['m'] = node_id, metadata
    return node


async def _embed_walk(node, doc_id, structural_path, file_path, collection,
                      embed_types, llm_client, model, written, skip_self_embed=False):
    # Hash before descendants acquire row annotations. Insertion remains child-first.
    node_id = _node_id(node, doc_id, structural_path) if isinstance(node, dict) else None
    if isinstance(node, dict):
        for key, value in list(node.items()):
            if isinstance(value, (dict, list)):
                node[key] = await _embed_walk(value, doc_id, _child_path(structural_path, key),
                    file_path, collection, embed_types, llm_client, model, written)
        if not skip_self_embed and node.get('t') in embed_types:
            await _persist_embedding(node, node_id, doc_id, structural_path, file_path,
                                     collection, llm_client, model, written)
    elif isinstance(node, list):
        for i, value in enumerate(node):
            if isinstance(value, (dict, list)):
                node[i] = await _embed_walk(value, doc_id, _child_path(structural_path, i),
                    file_path, collection, embed_types, llm_client, model, written)
    return node


def _wrapper_context(cur_object_path):
    # Legacy argument remains an ancestor-ID sequence, never a structural path.
    # First ancestor supplies document identity; additional ancestors are framed
    # into an explicit compatibility namespace. The sequence is never modified.
    doc_id = cur_object_path[0] if cur_object_path else ''
    path = '$' if len(cur_object_path) <= 1 else '$/legacy/' + hashlib.sha256(
        _canonical_json(cur_object_path).encode('utf-8')).hexdigest()
    return doc_id, path

async def embed(semantics_json: str, file_path: Path, embed_types: set[str],
                db_client: Optional[ClientAPI], collection: Collection,
                llm_client: AsyncClient, model: str='bge-m3',
                action: Optional[Callable]=None) -> str:
    """Replace this source's rows after a complete successful embedding walk.

    Upserts are not transactional: failure may leave additional new rows, but
    stale deletion happens only after all writes succeed. Retrying converges.
    Pre-provenance rows require a separate migration or collection rebuild.
    """
    # db_client is accepted for caller compatibility but unused.
    _validate_embed_types(embed_types)
    ast = json.loads(semantics_json)
    if not isinstance(ast, dict):
        raise ValueError('AST must be a dictionary')
    ast = _transform_embedding_tree(_semantic_payload(ast), action)
    source = str(file_path)
    doc_id = hashlib.sha256((source + _canonical_json(ast)).encode('utf-8')).hexdigest()
    written = set()
    try:
        old_ids = set(collection.get(where={'file_path': source}, include=[])['ids'])
        await _persist_embedding(ast, doc_id, doc_id, '$', source, collection,
                                 llm_client, model, written, root=True)
        await _embed_walk(ast, doc_id, '$', source, collection, embed_types,
                          llm_client, model, written, skip_self_embed=True)
        stale = old_ids - written
        if stale:
            collection.delete(ids=sorted(stale))
    except Exception as exc:
        raise RuntimeError(f'Embedding replacement failed for document {doc_id}, source {source}; '
                           f'{len(written)} rows upserted; retry to converge: {exc}') from exc
    return json.dumps(ast, ensure_ascii=False)


In [ ]:
# | hide
def test_embed_adds_ast_and_nested_nodes_then_returns_json():
    semantics = {
        "t": "Document",
        "s": "document summary",
        "children": [
            {"t": "Section", "s": "section summary", "children": []}
        ],
    }
    collection = FakeCollection()
    client = FakeAsyncClient([
        FakeEmbedResponse([[9, 9]], model="ast-model"),
        FakeEmbedResponse([[1, 1]], model="section-model"),
    ])

    result_json = run_async(
        embed(
            semantics_json=json.dumps(semantics),
            file_path=Path("sample.md"),
            embed_types={"Section"},
            db_client=None,
            collection=collection,
            llm_client=client,
            model="seed-model",
            action=lambda node: {**node, "seen": True} if isinstance(node, dict) else node,
        )
    )

    result = json.loads(result_json)
    test_eq(result["m"]["embed_model"], "ast-model")
    test_eq(result["m"]["node_type"], "Document")
    test_eq(result["m"]["document_id"], result["i"])
    test_eq(result["m"]["node_path"], "$")
    test_eq(result["m"]["file_path"], "sample.md")
    assert "seen" not in result["m"]
    test_eq(result["seen"], True)
    test_eq(result["children"][0]["seen"], True)
    test_eq(len(result["i"]), 64)
    test_eq(len(result["children"][0]["i"]), 64)
    test_eq(len(collection.records), 2)
    test_eq(collection.records[0]["documents"], ["document summary"])
    test_eq(collection.records[0]["metadatas"], [result["m"]])
    test_eq(collection.records[1]["documents"], ["section summary"])
    test_eq(collection.records[1]["metadatas"][0]["embed_model"], "section-model")
    test_eq(collection.records[1]["metadatas"][0]["node_type"], "Section")
    test_eq(collection.records[1]["metadatas"][0]["document_id"], result["i"])
    test_eq(collection.records[1]["metadatas"][0]["node_path"], "$/children/0")
    test_eq(collection.records[1]["metadatas"][0]["file_path"], "sample.md")
    test_eq(client.calls[0]["input"], "document summary")
    test_eq(client.calls[1]["input"], "section summary")

In [ ]:
# | export
async def embed_node(node: dict, collection: Collection, cur_object_path: list[str],
                     embed_types: set[str], llm_client: AsyncClient, model: str,
                     file_path: str='') -> dict:
    """Compatibility wrapper: ancestor IDs seed identity; standalone path is $.

    Missing source provenance is omitted. Explicit structural context is carried
    by the private pipeline helpers. Existing input IDs are never trusted.
    """
    _validate_embed_types(embed_types)
    if node.get('t') not in embed_types:
        raise ValueError(f"Node type {node.get('t')} is not in embed_types: {embed_types}")
    node = _semantic_payload(node)
    doc_id, path = _wrapper_context(cur_object_path)
    return await _persist_embedding(node, _node_id(node, doc_id, path), doc_id, path,
                                   str(file_path), collection, llm_client, model, set())


In [ ]:
# | hide
def test_embed_node_adds_collection_record_for_supported_node():
    node = {"t": "Section", "s": "section summary"}
    collection = FakeCollection()
    client = FakeAsyncClient([FakeEmbedResponse([[0.5, 1.5]], model="embed-model")])

    result = run_async(
        embed_node(
            node=node,
            collection=collection,
            cur_object_path=["root-id"],
            embed_types={"Section"},
            llm_client=client,
            model="seed-model",
        )
    )

    expected_metadata = {
        "embed_model": "embed-model",
        "node_type": "Section",
        "document_id": "root-id",
        "node_path": "$",
    }
    test_eq(result["s"], "section summary")
    test_eq(result["m"], expected_metadata)
    test_eq(len(result["i"]), 64)
    test_eq(len(collection.records), 1)
    test_eq(collection.records[0]["documents"], ["section summary"])
    test_eq(collection.records[0]["metadatas"], [expected_metadata])
    test_eq(collection.records[0]["embeddings"].shape, (1, 2))


def test_embed_node_skips_empty_summary_and_rejects_wrong_type():
    collection = FakeCollection()
    client = FakeAsyncClient([])
    empty_node = {"t": "Section", "s": ""}

    result = run_async(
        embed_node(
            node=empty_node,
            collection=collection,
            cur_object_path=["root-id"],
            embed_types={"Section"},
            llm_client=client,
            model="seed-model",
        )
    )

    test_eq(result.get("m"), None)
    test_eq(len(collection.records), 0)
    test_eq(len(client.calls), 0)
    test_fail(
        lambda: run_async(
            embed_node(
                node={"t": "Para", "s": "nope"},
                collection=collection,
                cur_object_path=["root-id"],
                embed_types={"Section"},
                llm_client=client,
                model="seed-model",
            )
        ),
        contains="Node type Para is not in embed_types",
    )


def test_embed_node_handles_dict_response_model():
    collection = FakeCollection()
    client = FakeAsyncClient([{"embeddings": [[0.5, 1.5]], "model": "dict-model"}])

    result = run_async(
        embed_node(
            node={"t": "Section", "s": "section summary"},
            collection=collection,
            cur_object_path=["root-id"],
            embed_types={"Section"},
            llm_client=client,
            model="seed-model",
        )
    )

    test_eq(result["m"]["embed_model"], "dict-model")
    test_eq(len(collection.records), 1)


def test_embed_node_dict_response_without_model_falls_back_to_requested():
    collection = FakeCollection()
    client = FakeAsyncClient([{"embeddings": [[0.5, 1.5]]}])

    result = run_async(
        embed_node(
            node={"t": "Section", "s": "section summary"},
            collection=collection,
            cur_object_path=["root-id"],
            embed_types={"Section"},
            llm_client=client,
            model="seed-model",
        )
    )

    test_eq(result["m"]["embed_model"], "seed-model")


test_embed_node_adds_collection_record_for_supported_node()
test_embed_node_skips_empty_summary_and_rejects_wrong_type()
test_embed_node_handles_dict_response_model()
test_embed_node_dict_response_without_model_falls_back_to_requested()

In [ ]:
# | export
async def embed_walk_node(node: dict | list, cur_object_path: list[str], collection: Collection,
                          embed_types: set[str], llm_client: AsyncClient, model: str='bge-m3',
                          action: Optional[Callable]=None, file_path: str='',
                          skip_self_embed: bool=False) -> dict | list:
    """Compatibility walk with immutable ancestor seed and standalone path $.

    Action visits dict/list containers once, including list-valued properties;
    scalar fields remain unchanged. All transforms finish before any API call.
    """
    _validate_embed_types(embed_types)
    if not isinstance(node, (dict, list)):
        raise ValueError('node must be a dict or list')
    node = _transform_embedding_tree(_semantic_payload(node), action)
    doc_id, path = _wrapper_context(cur_object_path)
    return await _embed_walk(node, doc_id, path, str(file_path), collection,
                            embed_types, llm_client, model, set(), skip_self_embed)


In [ ]:
# | hide
def test_embed_walk_node_recurses_lists_and_applies_action():
    semantics = {
        "t": "Section",
        "s": "root summary",
        "children": [
            {"t": "Table", "s": "table summary", "rows": []},
            {"plain": "value"},
        ],
    }
    collection = FakeCollection()
    client = FakeAsyncClient([
        FakeEmbedResponse([[1, 0]], model="walk-model"),
        FakeEmbedResponse([[0, 1]], model="walk-model"),
    ])

    object_path = ["ast-id"]
    result = run_async(
        embed_walk_node(
            node=semantics,
            cur_object_path=object_path,
            collection=collection,
            embed_types={"Section", "Table"},
            llm_client=client,
            model="seed-model",
            action=lambda node: {**node, "visited": True} if isinstance(node, dict) else node,
        )
    )

    test_eq(result["visited"], True)
    test_eq(result["children"][0]["visited"], True)
    test_eq(result["children"][1]["visited"], True)
    test_eq(len(collection.records), 2)
    test_eq(collection.records[0]["documents"], ["table summary"])
    test_eq(collection.records[1]["documents"], ["root summary"])
    # the compatibility ancestor seed remains unchanged
    test_eq(object_path, ["ast-id"])
    # structural paths are independent of ancestor IDs
    test_eq(collection.records[0]["metadatas"][0]["node_path"], "$/children/0")
    test_eq(collection.records[0]["metadatas"][0]["document_id"], "ast-id")


test_embed_adds_ast_and_nested_nodes_then_returns_json()
test_embed_walk_node_recurses_lists_and_applies_action()


In [ ]:
# | hide
def test_embed_walk_node_dict_child_embedded_once():
    semantics = {
        "t": "Section",
        "s": "root summary",
        "meta_node": {"t": "Table", "s": "table summary"},
    }
    collection = FakeCollection()
    client = FakeAsyncClient([
        FakeEmbedResponse([[1, 0]], model="walk-model"),
        FakeEmbedResponse([[0, 1]], model="walk-model"),
    ])

    result = run_async(
        embed_walk_node(
            node=semantics,
            cur_object_path=["ast-id"],
            collection=collection,
            embed_types={"Section", "Table"},
            llm_client=client,
            model="seed-model",
            action=lambda node: {**node, "visited": True} if isinstance(node, dict) else node,
        )
    )

    # regression: the dict-valued child must be embedded exactly once (child first, parent last)
    test_eq(len(collection.records), 2)
    test_eq(len(client.calls), 2)
    test_eq(collection.records[0]["documents"], ["table summary"])
    test_eq(collection.records[1]["documents"], ["root summary"])
    ids = [record["ids"][0] for record in collection.records]
    test_eq(len(set(ids)), 2)
    # action ran before embedding, and the surviving id is the one stored in the collection
    test_eq(result["meta_node"]["visited"], True)
    test_eq(result["meta_node"]["i"], ids[0])


def test_embed_walk_node_ancestor_path_and_determinism():
    def make_ast():
        return {
            "t": "Section",
            "s": "outer summary",
            "children": [
                {
                    "t": "Section",
                    "s": "inner summary",
                    "children": [{"t": "Table", "s": "table summary"}],
                },
            ],
        }

    def walk():
        collection = FakeCollection()
        client = FakeAsyncClient([FakeEmbedResponse([[1, 1]], model="walk-model")] * 3)
        object_path = ["ast-id"]
        result = run_async(
            embed_walk_node(
                node=make_ast(),
                cur_object_path=object_path,
                collection=collection,
                embed_types={"Section", "Table"},
                llm_client=client,
                model="seed-model",
            )
        )
        test_eq(object_path, ["ast-id"])
        return result, collection

    result, collection = walk()
    outer_i = result["i"]
    inner_i = result["children"][0]["i"]
    table_metadata = collection.records[0]["metadatas"][0]
    test_eq(collection.records[0]["documents"], ["table summary"])
    # node_path records dict keys and list indexes
    test_eq(table_metadata["node_path"], "$/children/0/children/0")
    test_eq(table_metadata["document_id"], "ast-id")
    test_eq(collection.records[1]["metadatas"][0]["node_path"], "$/children/0")
    test_eq(collection.records[2]["metadatas"][0]["node_path"], "$")

    # ids are deterministic across walks of identical content
    result2, collection2 = walk()
    test_eq(result2["i"], outer_i)
    test_eq([r["ids"] for r in collection2.records], [r["ids"] for r in collection.records])


def test_embed_root_embedded_once_when_root_type_is_embeddable():
    semantics = {"t": "Section", "s": "root summary", "children": []}
    collection = FakeCollection()
    client = FakeAsyncClient([FakeEmbedResponse([[2, 2]], model="root-model")])

    result_json = run_async(
        embed(
            semantics_json=json.dumps(semantics),
            file_path=Path("sample.md"),
            embed_types={"Section"},
            db_client=None,
            collection=collection,
            llm_client=client,
            model="seed-model",
        )
    )

    result = json.loads(result_json)
    canonical_json = json.dumps(
        semantics, sort_keys=True, ensure_ascii=False, separators=(",", ";")
    ).encode("utf-8").decode("utf-8")
    expected_id = hashlib.sha256(("sample.md" + canonical_json).encode("utf-8")).hexdigest()
    # regression: root keeps the id assigned by embed() and is embedded exactly once
    test_eq(result["i"], expected_id)
    test_eq(len(collection.records), 1)
    test_eq(len(client.calls), 1)
    test_eq(collection.records[0]["ids"], [expected_id])
    test_eq(collection.records[0]["metadatas"][0]["node_type"], "Section")


def test_embed_skips_empty_root_summary():
    semantics = {"summary": "", "blocks": [{"t": "Section", "s": "section summary", "children": []}]}
    collection = FakeCollection()
    client = FakeAsyncClient([FakeEmbedResponse([[3, 3]], model="section-model")])

    run_async(
        embed(
            semantics_json=json.dumps(semantics),
            file_path=Path("sample.md"),
            embed_types={"Section"},
            db_client=None,
            collection=collection,
            llm_client=client,
            model="seed-model",
        )
    )

    test_eq(len(client.calls), 1)
    test_eq(client.calls[0]["input"], "section summary")
    test_eq(len(collection.records), 1)
    test_eq(collection.records[0]["documents"], ["section summary"])


test_embed_walk_node_dict_child_embedded_once()
test_embed_walk_node_ancestor_path_and_determinism()
test_embed_root_embedded_once_when_root_type_is_embeddable()
test_embed_skips_empty_root_summary()

In [ ]:
# | hide
def test_embedding_replacement_identity_and_failure_contract():
    original = {'summary': '文档', 'blocks': [
        {'t': 'Section', 's': 'same'}, {'t': 'Section', 's': 'same'}],
        'a/~b': {'t': 'Table', 's': '表格'},
        'business': {'i': 'semantic', 'm': {'meaning': 'keep'}}}
    collection = FakeCollection()
    def execute(tree, action=None, target=None):
        target = collection if target is None else target
        client = FakeAsyncClient([{'embeddings': [[1, 2]], 'model': 'reported'}] * 4)
        result = json.loads(run_async(embed(json.dumps(tree), Path('sample.md'),
            {'Section', 'Table'}, None, target, client, 'requested', action)))
        return result, client
    result, client = execute(original)
    test_eq(len(collection.rows), 4)
    test_eq(len(client.calls), 4)
    assert result['blocks'][0]['i'] != result['blocks'][1]['i']
    test_eq(result['blocks'][0]['m']['node_path'], '$/blocks/0')
    test_eq(result['a/~b']['m']['node_path'], '$/a~1~0b')
    test_eq(result['m']['embed_model'], 'reported')
    test_eq(_semantic_payload(result), original)
    test_eq(result['i'], hashlib.sha256(('sample.md'+_canonical_json(original)).encode()).hexdigest())
    repeated, _ = execute(result)
    test_eq(repeated, result)
    test_eq(len(collection.rows), 4)
    changed = {'summary': 'replacement', 'blocks': [{'t':'Section','s':'new'}]}
    replacement, _ = execute(changed)
    test_eq(set(collection.rows), {replacement['i'], replacement['blocks'][0]['i']})
    test_eq(collection.operations[-1][0], 'delete')

    seen = []
    def transform(node):
        seen.append(type(node).__name__)
        if isinstance(node, dict):
            return {**node, 'seen': True}
        return node
    transformed, _ = execute(changed, transform)
    test_eq(seen, ['dict','list','dict'])
    test_eq(transformed['i'], hashlib.sha256(('sample.md'+_canonical_json(
        {'summary':'replacement','seen':True,'blocks':[{'t':'Section','s':'new','seen':True}]})).encode()).hexdigest())
    untouched = FakeCollection()
    test_fail(lambda: execute(changed, lambda node: [] if isinstance(node,dict) else node, untouched),
              contains='action must preserve dict')
    test_eq(untouched.operations, [])
    empty, empty_client = execute({'summary':'  ','blocks':[]}, target=FakeCollection())
    assert 'i' not in empty and 'm' not in empty
    test_eq(empty_client.calls, [])

    for stage in ('get','upsert','delete','provider'):
        failing = FakeCollection()
        execute(original, target=failing)
        old_ids = set(failing.rows)
        original_method = getattr(failing, stage) if stage != 'provider' else None
        def fail(*args, **kwargs):
            raise RuntimeError('injected failure')
        if original_method is not None:
            setattr(failing, stage, fail)
            test_fail(lambda: execute(changed, target=failing), contains='retry to converge')
            setattr(failing, stage, original_method)
        else:
            failing_client = FakeAsyncClient([{'embeddings': []}, {'embeddings': []}])
            test_fail(lambda: run_async(embed(json.dumps(changed), Path('sample.md'),
                {'Section'}, None, failing, failing_client)), contains='retry to converge')
        assert old_ids <= set(failing.rows)
        recovered, _ = execute(changed, target=failing)
        test_eq(set(failing.rows), {recovered['i'], recovered['blocks'][0]['i']})


test_embedding_replacement_identity_and_failure_contract()


# Embedding Function Test

Run the **Summary** tests in **DomClass Function Tests** in
`01.core.dom.model.ipynb` first. These manual integration tests reuse the generated
`SN024002_semantics.json` and `SX022001_semantics.json` artifacts to embed each DOM
with Ollama's `bge-m3` model. They require a running Ollama service with that model
available and persist vectors in the project's `db` directory, in the
`mitochondria` collection. Each run replaces the source document's embeddings and
writes its `_embed.json` artifact without changing its AST or semantics.


In [ ]:
# | hide
# | notest
from chromadb import PersistentClient
from ribosome.core.dom.model import DOMClass
from ribosome.system.utils import get_project_root


async def _embed_function_test_dom(asset_name):
    project_root = get_project_root()
    md_file = ((project_root / "assets" / asset_name / f"{asset_name}.md")
               if project_root is not None else Path("assets") / asset_name / f"{asset_name}.md")
    if not md_file.exists():
        md_file = Path("../assets") / asset_name / f"{asset_name}.md"
    assert md_file.is_file(), f"Markdown source missing: {md_file}"
    dom = DOMClass(md_file)
    dom._sync_artifact_fields()
    assert dom.ast_json_file and dom.ast_json_file.is_file(), (
        "Run the DomClass Function Tests in 01.core.dom.model.ipynb first"
    )
    assert dom.semantics_json_file and dom.semantics_json_file.is_file(), (
        "Run the Summary tests in 01.core.dom.model.ipynb first"
    )
    ast_before = dom.ast_json_file.read_bytes()
    semantics_before = dom.semantics_json_file.read_bytes()
    dom.ast_json = ast_before.decode("utf-8")
    dom.semantics_json = semantics_before.decode("utf-8")
    semantics = json.loads(dom.semantics_json)
    assert semantics.get("blocks"), "Semantic DOM blocks missing"

    db_path = (project_root / "db" if project_root is not None
               else md_file.resolve().parents[2] / "db")
    dom.db_client = PersistentClient(path=str(db_path))
    collection = dom.db_client.get_or_create_collection(name="mitochondria")
    print(f"Starting embedding: {dom.semantics_json_file}")
    # Call this notebook's implementation, even when an older embed cache exists.
    dom.embed_json = await embed(
        semantics_json=dom.semantics_json,
        file_path=dom.file_path,
        embed_types=dom.Embed_Types,
        db_client=dom.db_client,
        collection=collection,
        llm_client=dom.ollama_client,
        model="bge-m3",
    )
    embedded = json.loads(dom.embed_json)
    test_eq(_semantic_payload(embedded), semantics)

    def embedded_nodes(node):
        if isinstance(node, dict):
            if isinstance(node.get("m"), dict) and "embed_model" in node["m"]:
                yield node
            for value in node.values():
                yield from embedded_nodes(value)
        elif isinstance(node, list):
            for value in node:
                yield from embedded_nodes(value)

    nodes = list(embedded_nodes(embedded))
    assert nodes, "No summaries were embedded"
    rows = collection.get(where={"file_path": str(dom.file_path)},
                          include=["metadatas", "documents"])
    test_eq(set(rows["ids"]), {node["i"] for node in nodes})
    for node in nodes:
        index = rows["ids"].index(node["i"])
        test_eq(rows["metadatas"][index], node["m"])
        test_eq(rows["documents"][index],
                node.get("s") or node.get("summary", ""))

    assert dom.embed_json_file, "Embedding artifact path missing"
    dom.embed_json_file.write_text(dom.embed_json, encoding="utf-8")
    test_eq(json.loads(dom.embed_json_file.read_text(encoding="utf-8")), embedded)
    test_eq(dom.ast_json_file.read_bytes(), ast_before)
    test_eq(dom.semantics_json_file.read_bytes(), semantics_before)
    print(f"Embedded {len(nodes)} summaries: {dom.embed_json_file}")
    return dom


## SN024002


In [ ]:
# | hide
# | notest
async def test_embed_with_sn024002_md():
    return await _embed_function_test_dom("SN024002")


dom_sn024002 = await test_embed_with_sn024002_md()


## SX022001


In [ ]:
# | hide
# | notest
async def test_embed_with_sx022001_md():
    return await _embed_function_test_dom("SX022001")


dom_sx022001 = await test_embed_with_sx022001_md()


In [ ]:
# | hide
# | notest
import nbdev

nbdev.nbdev_export()